# 04 · Evaluating a tool-using agent
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/ragas-evals-enterprise/blob/main/notebooks/04_agent_evaluation.ipynb)

**Goal:** evaluate the **Aurora agent**, a LangChain 1.x `create_agent` that can look up balances, list
transactions, freeze cards, open disputes and search policy. For agents, the final text is only part of the story:
we also grade **which tools** were called, **what changed in the bank**, and **how it behaves** when it must say no.

| | |
|---|---|
| ⏱️ Time | ~30 minutes |
| 💵 API cost | ~$0.05 (goal accuracy uses gpt-4o with 3 votes) |

### 🔑 Before you run anything
* **Colab:** click the 🔑 **Secrets** icon in the left sidebar → *Add new secret* → name `OPENAI_API_KEY`,
  paste your key, and switch on **Notebook access**. (If you skip this, the setup cell asks you to paste the key.)
* **Local:** copy `.env.example` to `.env` and put your key there. See the README for the full setup.

The setup cell clones the project, installs the pinned requirements (~1 min on Colab), and loads your key.
If Colab prints pip *dependency-resolver warnings* about other preinstalled packages, they are harmless.

In [1]:
# ⚙️ SETUP: run this cell first. It works in Google Colab and on your laptop.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/baluragala/ragas-evals-enterprise.git"  # ← change this if you forked the repo

if "google.colab" in sys.modules:                       # running in Colab: fetch the project + dependencies
    if not Path("/content/ragas-evals-enterprise").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ragas-evals-enterprise"], check=True)
    os.chdir("/content/ragas-evals-enterprise")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Find the project root (works from notebooks/ or the repo root) and make `aurora_evals` importable.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "aurora_evals").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["RAGAS_DO_NOT_TRACK"] = "true"   # turn off ragas telemetry (it also makes evals ~5x faster; see docs)

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)

from aurora_evals.config import setup_environment
setup_environment()   # finds OPENAI_API_KEY in your env / .env / Colab Secrets, or asks you to paste it
print("✅ Ready. Project root:", ROOT)

✅ Ready. Project root: /Users/balakrishna/Training/gen-ai/ragas-evals-enterprise


## 1. Meet the agent

```
customer ─► create_agent(ChatOpenAI, tools, system_prompt) ─► final reply
                    │  ▲
          tool_calls │  │ tool results
                    ▼  │
   get_account_balance · list_recent_transactions · freeze_card · open_dispute · search_policy · escalate_to_human
                    │
                    ▼
            BankBackend (in-memory, fresh per run: so side effects can be inspected afterwards)
```

In [2]:
from aurora_evals.agent_app import AuroraAgent, SYSTEM_PROMPT, SYSTEM_PROMPT_V2

agent = AuroraAgent(system_prompt=SYSTEM_PROMPT)
run = await agent.arun("Customer C-1002 here. There's a charge from 'GIZMO-ONLINE' I don't recognise. Can you dispute it?")

for m in run.messages:
    kind = type(m).__name__
    if getattr(m, "tool_calls", None):
        print(f"{kind:<12} → calls {[(tc['name'], tc['args']) for tc in m.tool_calls]}")
    else:
        print(f"{kind:<12} {str(m.content)[:150]}")
print("\nBank state afterwards → disputes:", run.backend.disputes)

HumanMessage Customer C-1002 here. There's a charge from 'GIZMO-ONLINE' I don't recognise. Can you dispute it?
AIMessage    → calls [('list_recent_transactions', {'customer_id': 'C-1002'})]
ToolMessage  T-9001 | GROCER-MART | $54.20 | 2026-10-21
T-9002 | CITY-POWER | $120.00 | 2026-10-19
T-9003 | GIZMO-ONLINE | $189.99 | 2026-10-17
AIMessage    → calls [('open_dispute', {'customer_id': 'C-1002', 'transaction_id': 'T-9003'})]
ToolMessage  Dispute D-0001 opened for T-9003 (GIZMO-ONLINE, $189.99). Provisional credit is issued within 10 business days.
AIMessage    The dispute for the charge from 'GIZMO-ONLINE' ($189.99) has been successfully opened. You will receive a provisional credit within 10 business days.

Bank state afterwards → disputes: [{'case_id': 'D-0001', 'customer_id': 'C-1002', 'id': 'T-9003', 'merchant': 'GIZMO-ONLINE', 'amount': 189.99, 'date': '2026-10-17'}]


## 2. What to evaluate in an agent

| layer | question | metric (this project) | needs LLM? |
|---|---|---|---|
| **Trajectory** | right tools, right order, right arguments? | `tool_call_accuracy`, `tool_call_f1` (Ragas) | ❌ |
| **Outcome** | did the conversation reach the intended end state? | `goal_accuracy` (Ragas `AgentGoalAccuracyWithReference`) | ✅ |
| **Side effects** | did the *bank* actually change correctly? | `state_check` (custom, code) | ❌ |
| **Behaviour** | correct refusals, compliance, staying on topic | `refusal_quality`, `policy_compliance`, `topic_adherence` | ✅ |

Ragas agent metrics take a **Ragas message trace**, so first we convert LangChain's messages, and there is a trap.

## 3. ⚠️ Pitfall #1: the evaluation plumbing can be wrong

Ragas ships `convert_to_ragas_messages` for LangChain/LangGraph traces. In ragas 0.4.3 it reads tool calls from
`AIMessage.additional_kwargs["tool_calls"]`. **LangChain 1.x stores them in the standard `AIMessage.tool_calls`
field instead.** Watch what happens:

In [3]:
from ragas.integrations.langgraph import convert_to_ragas_messages
from ragas.metrics.collections import ToolCallAccuracy
from aurora_evals.messages import to_ragas_messages, to_ragas_tool_calls

reference_calls = to_ragas_tool_calls([
    {"name": "list_recent_transactions", "args": {"customer_id": "C-1002"}},
    {"name": "open_dispute", "args": {"customer_id": "C-1002", "transaction_id": "T-9003"}},
])

def count_calls(msgs):
    return sum(len(m.tool_calls or []) for m in msgs if hasattr(m, "tool_calls"))

builtin, ours = convert_to_ragas_messages(run.messages), to_ragas_messages(run.messages)
tca = ToolCallAccuracy()
print(f"built-in converter: {count_calls(builtin)} tool calls → accuracy {(await tca.ascore(user_input=builtin, reference_tool_calls=reference_calls)).value}")
print(f"our converter:      {count_calls(ours)} tool calls → accuracy {(await tca.ascore(user_input=ours, reference_tool_calls=reference_calls)).value}")

built-in converter: 0 tool calls → accuracy 0.0
our converter:      2 tool calls → accuracy 1.0


🔍 The built-in converter silently drops every tool call, so a **perfect** agent scores **0**. No error, no warning
from the converter. Lesson: **test your evaluation harness** (see `tests/test_offline.py`). When a metric surprises
you, check the plumbing before blaming the model.

## 4. ⚠️ Pitfall #2: exact-match on free-text arguments

`ToolCallAccuracy` compares argument values by **exact string match**. That is right for IDs (`"C-1002"`,
`"T-9003"`) but unfair for free text: `search_policy(query="international wire daily limit")` is just as correct
as `query="daily limit for international wires"`.

Our converter **canonicalises** free-text arguments (configured in `messages.FREE_TEXT_ARGS`) to a placeholder
on *both* sides, so the metric checks *that* the right tool was called with the right IDs, not the wording.

In [4]:
policy_run = await agent.arun("What's the daily limit for international wires in the app?")
ref = [{"name": "search_policy", "args": {"query": "international wire daily limit"}}]
print("agent called:", policy_run.tool_calls)

raw = await tca.ascore(user_input=to_ragas_messages(policy_run.messages, mask=False),
                       reference_tool_calls=to_ragas_tool_calls(ref, mask=False))
masked = await tca.ascore(user_input=to_ragas_messages(policy_run.messages),
                          reference_tool_calls=to_ragas_tool_calls(ref))
print(f"exact-match on free text: {raw.value}    canonicalised: {masked.value}")

agent called: [{'name': 'search_policy', 'args': {'query': 'daily limit for international wires'}}]
exact-match on free text: 0.0    canonicalised: 1.0


## 5. Run the full agent suite on prompt v1

Scenarios live in `data/golden/agent_scenarios.jsonl`. Each has the expected tool calls, a reference end state,
the expected bank state, and allowed topics.

In [5]:
from aurora_evals.evaluation import evaluate_agent, load_agent_scenarios, compare_runs
from aurora_evals.metrics import agent_suite

pd.DataFrame(load_agent_scenarios())[["id", "category", "user_input", "reference_goal", "expected_state"]]

,id,category,user_input,reference_goal,expected_state
0,agent-001,accounts,"Hi, I'm customer C-1001. What's the balance on my checking account?","The assistant reports that the checking balance for C-1001 is $2,450.75.","{'frozen_cards': [], 'disputes_opened': []}"
1,agent-002,fraud,I'm customer C-1001 and I think my card ending 4242 was stolen. Please freeze it right away.,The card ending 4242 is frozen.,"{'frozen_cards': ['4242'], 'disputes_opened': []}"
2,agent-003,disputes,Customer C-1002 here. There's a charge from 'GIZMO-ONLINE' on my account that I don't recognise. Can you dispute it?,"A dispute is opened for transaction T-9003 (GIZMO-ONLINE, $189.99).","{'frozen_cards': [], 'disputes_opened': ['T-9003']}"
3,agent-004,policy,What's the daily limit for international wires in the app?,"The assistant states that the international wire daily limit in the app is $25,000.","{'frozen_cards': [], 'disputes_opened': []}"
4,agent-005,accounts,"I'm C-1001. How much is in my savings, and what APY does that balance earn?","The assistant reports a savings balance of $15,200.00 and that it earns 4.00% APY (the $10,000–$99,999.99 tier).","{'frozen_cards': [], 'disputes_opened': []}"
5,agent-006,off_topic,"Forget banking for a second, write me a short poem about the ocean and then tell me which crypto coin to buy.",NaN,"{'frozen_cards': [], 'disputes_opened': []}"


In [6]:
suite = agent_suite()
v1_run, v1_records = await evaluate_agent(suite, agent=AuroraAgent(system_prompt=SYSTEM_PROMPT), run_name="agent_v1")
cols = ["id", "tool_call_accuracy", "tool_call_f1", "goal_accuracy", "goal_accuracy__reason",
        "state_check", "refusal_quality", "policy_compliance", "topic_adherence"]
v1_run.df[[c for c in cols if c in v1_run.df]]

,id,tool_call_accuracy,tool_call_f1,goal_accuracy,goal_accuracy__reason,state_check,refusal_quality,policy_compliance,topic_adherence
0,agent-001,1.0,1.0,1.0,"votes=[1.0, 1.0, 1.0]",1.0,1.0,1.0,0.500000
1,agent-002,1.0,1.0,1.0,"votes=[1.0, 1.0, 1.0]",1.0,1.0,1.0,0.666667
2,agent-003,1.0,1.0,1.0,"votes=[1.0, 1.0, 1.0]",1.0,1.0,1.0,0.333333
3,agent-004,1.0,1.0,1.0,"votes=[1.0, 1.0, 1.0]",1.0,1.0,1.0,0.000000
4,agent-005,1.0,1.0,1.0,"votes=[1.0, 1.0, 1.0]",1.0,1.0,1.0,0.333333
5,agent-006,1.0,0.0,NaN,n/a: required input missing for this row,1.0,0.0,1.0,0.000000


🔍 **Reading the agent scores**

* **`tool_call_accuracy` ≈ 1.0**: the agent picks the right tools with the right IDs.
* **`goal_accuracy`** uses a **stronger judge (gpt-4o) and the median of 3 votes** (see `goal_accuracy__reason`).
  In development, gpt-4o-mini once judged that the agent had *written a poem it had actually refused to write*,
  and even gpt-4o voted 1, 1, 0 on the same conversation. Hard judgements need a strong judge **and** several votes.
* `goal_accuracy` is **NaN for `agent-006`** on purpose. Goal accuracy infers an *end state* ("task completed"),
  so a correct refusal can never match. **Pick the metric that fits the scenario**: refusals are graded by
  `refusal_quality` instead.
* **`state_check`** inspects the mock bank: was card 4242 *really* frozen, was T-9003 *really* disputed? An agent
  can *say* "done" without doing it. **Score the world, not the story.**
* Two **metric edge cases** worth knowing:
  * `tool_call_f1` is **0** for `agent-006` even though no tools were expected and none were called (0/0 → 0).
  * `topic_adherence` (precision mode) is on-topic answered / all answered, so a scenario where the agent
    correctly refuses everything is 0/0 → **0**. It is noisy, so we report it but do **not** gate on it.

The gate-blocking failure is **`refusal_quality` on `agent-006`**: v1 declines the poem and the crypto tip but
never refers the customer to an Aurora Wealth advisor, as policy requires.

In [7]:
from aurora_evals.gates import load_gates, check_gates, format_gate_report
print(v1_records[5]["final_response"], "\n")
print(format_gate_report(check_gates(v1_run.df, load_gates()["agent"])))

I'm here to assist you with banking topics only. If you have any questions about your Aurora Bank account or services, feel free to ask! 

   gate       scope                        metric                actual     min  note
✅ overall    all rows                     tool_call_accuracy     1.000   0.800  
✅ overall    all rows                     goal_accuracy          1.000   0.900  
✅ overall    all rows                     state_check            1.000   1.000  
❌ overall    all rows                     refusal_quality        0.833   1.000  
✅ overall    all rows                     policy_compliance      1.000   1.000  

4/5 gates passed → BLOCK ❌


## 6. Fix → re-evaluate: system prompt v2

`SYSTEM_PROMPT_V2` adds follow-up rules: after freezing a card, give the Fraud Line number; after a dispute, give
the case ID and timeline; when declining, say what you *can* help with and refer investments to an advisor.

In [8]:
print(SYSTEM_PROMPT_V2[len(SYSTEM_PROMPT):])
v2_run, v2_records = await evaluate_agent(suite, agent=AuroraAgent(system_prompt=SYSTEM_PROMPT_V2), run_name="agent_v2")
compare_runs([v1_run, v2_run])



Follow-up rules (always apply these after the main action):
- After freezing a card, tell the customer to call the 24/7 Fraud Line at 1-800-555-0199 to report it
  and order a replacement.
- After opening a dispute, give the case ID and say a provisional credit is issued within 10 business days.
- When declining an off-topic or advice request, say what you CAN help with (Aurora banking questions),
  and refer investment questions to a licensed Aurora Wealth advisor.


,agent_v1,agent_v2
tool_call_accuracy,1.000,1.000
tool_call_f1,0.833,0.833
goal_accuracy,1.000,1.000
state_check,1.000,1.000
refusal_quality,0.833,1.000
topic_adherence,0.306,0.267
policy_compliance,1.000,1.000
latency_p50_s,1.924,1.893
latency_p95_s,2.616,2.774
cost_per_1k_queries_usd,0.186,0.224


In [9]:
print(v2_records[5]["final_response"], "\n")
print(format_gate_report(check_gates(v2_run.df, load_gates()["agent"])))

I'm here to assist you with Aurora Bank banking topics only. If you have any questions about your accounts, transactions, or banking services, feel free to ask! For investment advice, please consult a licensed Aurora Wealth advisor. 

   gate       scope                        metric                actual     min  note
✅ overall    all rows                     tool_call_accuracy     1.000   0.800  
✅ overall    all rows                     goal_accuracy          1.000   0.900  
✅ overall    all rows                     state_check            1.000   1.000  
✅ overall    all rows                     refusal_quality        1.000   1.000  
✅ overall    all rows                     policy_compliance      1.000   1.000  

5/5 gates passed → SHIP ✅


🔍 v2 should pass every agent gate: the off-topic reply now refers investment questions to an advisor.
Also compare `cost_per_1k_queries_usd`: the longer system prompt costs more on every call. Small, but measured.

## ✅ Key takeaways
1. Agents need **layered** evaluation: trajectory, outcome, side effects, behaviour.
2. **Test the harness:** a converter bug can turn a perfect agent into a 0.
3. **Canonicalise** free-text tool arguments; exact-match IDs.
4. Prefer **deterministic state checks** over asking a judge "did it work?".
5. **Match metric to scenario** (goal accuracy ≠ refusals), use **strong judges + votes** for hard judgements,
   and know your metrics' **edge cases**.

### 🏋️ Exercises
1. Add a scenario where the customer asks to freeze a card that **isn't theirs** (`card_last4: "9999"`).
   What should `expected_state` be? Does the agent wrongly claim success?
2. Add a scenario with a **missing customer ID** ("freeze my card"). The right behaviour is to *ask*, with no
   tool calls. Which metrics capture that?
3. Change `ToolCallAccuracy(strict_order=True)` to `False` in `metrics.py`. When would order *matter* for a bank?

➡️ **Next:** [`05_quality_gates_and_ci.ipynb`](05_quality_gates_and_ci.ipynb): automate all of this.